# Chapitre 12 - Exercices 12.6 à 12.11

* 12.6 : *optimal scoring* et problème aux valeurs propres ;
* 12.7 : *optimal scoring* pénalisé (PDA) ;
* 12.8 : les coefficients de l'*optimal scoring* sont proportionnels aux directions de la LDA ;
* 12.9 : LDA sur les valeurs ajustées d'une régression sur indicatrices ;
* 12.10 : LDA régularisée à noyau ;
* 12.11 : un modèle de mélange à centres partagés (MDA2) et son algorithme EM.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch12_svm import (LDA, FDA, MDA, class_indicator, second_diff_penalty, kmeans,
                          rbf_kernel, poly2_features, polynomial_kernel, simulate_mixture,
                          mixture_bayes_prob, plot_mixture_boundary, simulate_waveform)
set_style()

Notations communes : $Y$ est la matrice indicatrice $N\times K$ des classes,
$D_\pi = Y^TY/N = \mathrm{diag}(\hat\pi_1,\dots,\hat\pi_K)$ la matrice des proportions de
classes, $H$ la matrice $N\times M$ des $h_j(x_i)$. Nous utilisons les données *vowel*
(11 classes, 10 variables) pour les vérifications.

In [2]:
Xv, gv = datasets.load_vowel("train")
Yv, cls = class_indicator(gv)
N, K = Yv.shape
Dpi = Yv.T @ Yv / N
Hv = np.column_stack([np.ones(N), Xv])          # base : constante + variables
print("N = %d, K = %d, p = %d" % (N, K, Xv.shape[1]))

N = 528, K = 11, p = 10


## Exercice 12.6 : *Optimal scoring* et valeurs propres

**Énoncé (reformulé).** La régression de la FDA est un développement linéaire sur des
fonctions de base $h_m(x)$, $m = 1,\dots,M$.

(a) Montrer que le problème (12.52) s'écrit $\min_{\theta,\beta}\|Y\theta - H\beta\|^2$ (12.65),
où $\theta\in\mathbb R^K$.
(b) Interpréter les normalisations $\theta^TD_\pi\mathbf 1 = 0$ et $\theta^TD_\pi\theta = 1$.
(c) Montrer qu'après optimisation en $\beta$, on obtient $\max_\theta\theta^TY^TSY\theta$ (12.66)
sous ces contraintes, où $S$ est le projecteur associé à $H$.
(d) Si les $h_j$ contiennent la fonction constante, montrer que la plus grande valeur propre
de $S$ vaut 1.
(e) Avec une matrice $\Theta$ ($K\times K$) de scores normalisée par $\Theta^TD_\pi\Theta = I$,
montrer que la solution de (12.53) est donnée par l'ensemble des vecteurs propres ; le
premier est trivial et assure le centrage, les autres caractérisent la solution.

**(a)** Le score de l'observation $i$ est $\theta(g_i) = \sum_k y_{ik}\theta_k = (Y\theta)_i$
puisque la ligne $i$ de $Y$ ne contient qu'un 1, en colonne $g_i$. La prédiction est
$\sum_m\beta_mh_m(x_i) = (H\beta)_i$. Donc $\sum_i(\theta(g_i) - h(x_i)^T\beta)^2 = \|Y\theta - H\beta\|^2$.

**(b)** $\theta^TD_\pi\mathbf 1 = \sum_k\hat\pi_k\theta_k = \frac1N\sum_i\theta(g_i)$ : les
scores sont **centrés** sur l'échantillon. $\theta^TD_\pi\theta = \frac1N\sum_i\theta(g_i)^2$ :
avec le centrage, c'est la **variance** empirique des scores, fixée à 1. Ces contraintes
excluent la solution triviale $\theta = 0$ (et la solution constante).

**(c)** À $\theta$ fixé, (12.65) est un problème de moindres carrés en $\beta$ : $\hat\beta =
(H^TH)^{-1}H^TY\theta$ et $H\hat\beta = SY\theta$ avec $S = H(H^TH)^{-1}H^T$, projecteur
orthogonal ($S^T = S$, $S^2 = S$). Le résidu vaut

$$\|(I-S)Y\theta\|^2 = \theta^TY^T(I-S)^T(I-S)Y\theta = \theta^TY^T(I-S)Y\theta = N\theta^TD_\pi\theta - \theta^TY^TSY\theta = N - \theta^TY^TSY\theta,$$

en utilisant $(I-S)^T(I-S) = I - S$ et $Y^TY = ND_\pi$. Minimiser le résidu revient donc à
maximiser $\theta^TY^TSY\theta$ sous les contraintes de (b).

**(d)** $S$ est un projecteur : si $Sv = \mu v$ avec $v\ne0$, alors $\mu v = Sv = S^2v = \mu^2v$,
donc $\mu\in\{0, 1\}$. Si la constante est dans l'espace engendré par les colonnes de $H$,
alors $S\mathbf 1 = \mathbf 1$ : 1 est valeur propre, c'est la plus grande.

Version « problème d'optimal scoring » : le quotient $\frac{\theta^TY^TSY\theta}{N\theta^TD_\pi\theta} = \frac{\|SY\theta\|^2}{\|Y\theta\|^2}\le1$
(une projection orthogonale diminue la norme), et il vaut 1 pour $\theta = \mathbf 1$ car
$Y\mathbf 1 = \mathbf 1$ et $S\mathbf 1 = \mathbf 1$. La plus grande valeur propre généralisée de
$(\frac1NY^TSY, D_\pi)$ vaut donc 1, atteinte par les scores constants.

**(e)** Avec $L$ colonnes de scores, (12.53) se réduit de même à maximiser
$\mathrm{tr}(\Theta^TY^TSY\Theta)$ sous $\Theta^TD_\pi\Theta = I$. Posons $\Phi = D_\pi^{1/2}\Theta$ :
la contrainte devient $\Phi^T\Phi = I$ et l'objectif $\mathrm{tr}(\Phi^TA\Phi)$ avec
$A = D_\pi^{-1/2}Y^TSYD_\pi^{-1/2}$ symétrique. Le maximum d'une trace sous contrainte
d'orthonormalité est atteint par les vecteurs propres de $A$ associés aux plus grandes valeurs
propres (théorème de Ky Fan, généralisation du quotient de Rayleigh). Revenant à $\Theta$ : les
colonnes de $\Theta$ sont les vecteurs propres **généralisés**
$Y^TSY\theta = \mu\,ND_\pi\theta$.

* Le premier ($\mu = 1$) est $\theta = \mathbf 1$ (normalisé : $\mathbf 1^TD_\pi\mathbf 1 = 1$) :
  solution triviale.
* Les autres sont $D_\pi$-orthogonaux au premier : $\theta^TD_\pi\mathbf 1 = 0$, c'est-à-dire
  **centrés** - la contrainte (b) est automatiquement satisfaite. Ce sont les scores optimaux.

**Vérification numérique** sur *vowel* avec $H = [\mathbf 1, X]$.

In [3]:
S = Hv @ np.linalg.solve(Hv.T @ Hv, Hv.T)
print("valeurs propres de S dans {0,1} :", np.allclose(np.sort(np.linalg.eigvalsh(S))[-(Hv.shape[1] + 1):].round(8),
                                                     np.r_[0, np.ones(Hv.shape[1])]))
mu, Theta = eigh(Yv.T @ S @ Yv / N, Dpi)
mu, Theta = mu[::-1], Theta[:, ::-1]
print("valeurs propres généralisées :", mu.round(4))
print("premier vecteur propre (constant ?) :", (Theta[:, 0] * np.sign(Theta[0, 0])).round(4))
print("Theta^T Dpi Theta = I :", np.allclose(Theta.T @ Dpi @ Theta, np.eye(K)))
print("scores suivants centrés (theta^T Dpi 1 = 0) :", np.allclose(Theta[:, 1:].T @ Dpi @ np.ones(K), 0))

valeurs propres de S dans {0,1} : True
valeurs propres généralisées : [1.000e+00 8.021e-01 7.174e-01 2.432e-01 1.213e-01 7.140e-02 5.650e-02
 1.830e-02 7.600e-03 1.000e-03 6.000e-04]
premier vecteur propre (constant ?) : [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Theta^T Dpi Theta = I : True
scores suivants centrés (theta^T Dpi 1 = 0) : True


Il y a $\min(K-1, p) = 10$ valeurs propres non triviales non nulles ; la dernière est nulle car
$p + 1 = 11 = K$. Vérifions enfin que le résidu moyen vaut bien $1 - \mu_\ell$ pour chaque score :

In [4]:
for l in [1, 2, 3]:
    th = Theta[:, l]
    resid = Yv @ th - S @ Yv @ th
    print("score %d : ASR = %.4f   1 - mu = %.4f" % (l, resid @ resid / N, 1 - mu[l]))

score 1 : ASR = 0.1979   1 - mu = 0.1979
score 2 : ASR = 0.2826   1 - mu = 0.2826
score 3 : ASR = 0.7568   1 - mu = 0.7568


> **À retenir.** L'*optimal scoring* est un problème aux valeurs propres généralisé
> $Y^TSY\theta = \mu ND_\pi\theta$ ; la solution constante ($\mu = 1$) absorbe le centrage.

## Exercice 12.7 : *Optimal scoring* pénalisé

**Énoncé.** Obtenir la solution du problème d'*optimal scoring* pénalisé (12.57) :

$$\min\ \frac1N\sum_{\ell}\left[\sum_i(\theta_\ell(g_i) - h(x_i)^T\beta_\ell)^2 + \lambda\beta_\ell^T\Omega\beta_\ell\right].$$

**Solution.** *Étape 1 : optimisation en $\beta$ à $\theta$ fixé.* Pour un score, le critère
$\|Y\theta - H\beta\|^2 + \lambda\beta^T\Omega\beta$ est quadratique convexe en $\beta$ ; son
gradient $-2H^T(Y\theta - H\beta) + 2\lambda\Omega\beta$ s'annule pour

$$\hat\beta = (H^TH + \lambda\Omega)^{-1}H^TY\theta\qquad\text{(régression ridge généralisée)}.$$

*Étape 2 : valeur du critère.* En notant $A = H^TH + \lambda\Omega$, on développe et on utilise
$A\hat\beta = H^TY\theta$ :

$$\|Y\theta - H\hat\beta\|^2 + \lambda\hat\beta^T\Omega\hat\beta = \theta^TY^TY\theta - 2\hat\beta^TH^TY\theta + \hat\beta^TA\hat\beta = \theta^TY^TY\theta - \theta^TY^TS_\lambda Y\theta,$$

avec $S_\lambda = H(H^TH + \lambda\Omega)^{-1}H^T$ (l'opérateur de lissage, qui n'est plus un
projecteur).

*Étape 3 : scores.* Avec la normalisation $\theta^TD_\pi\theta = 1$, minimiser revient à
maximiser $\theta^TY^TS_\lambda Y\theta$ : les scores optimaux sont les vecteurs propres
généralisés de $(\frac1NY^TS_\lambda Y, D_\pi)$, exactement comme en 12.6(e) avec $S_\lambda$ à
la place de $S$. Si la constante est dans la base et **non pénalisée**, $S_\lambda\mathbf 1 = \mathbf 1$
(le vecteur $\beta = e_1$ vérifie $A e_1 = H^T\mathbf 1$ car $\Omega e_1 = 0$) : la solution
triviale subsiste et les autres scores sont centrés. Enfin $\hat\beta_\ell = A^{-1}H^TY\theta_\ell$.

*Étape 4 : lien avec la PDA.* Séparons la constante et notons $\tilde H$ la matrice des
$h(x_i)$ centrés, $\Sigma_T = \tilde H^T\tilde H/N$, $M$ la matrice $K\times M$ des moyennes de
classes centrées, $\Sigma_B = M^TD_\pi M$ et $\Sigma_W = \Sigma_T - \Sigma_B$. Alors
$\tilde H^TY = NM^TD_\pi$ et $u = \hat\beta = (\Sigma_T + \frac\lambda N\Omega)^{-1}M^TD_\pi\theta$.
L'équation aux valeurs propres $D_\pi M u = \mu D_\pi\theta$ donne $Mu = \mu\theta$, d'où
$\Sigma_Bu = M^TD_\pi Mu = \mu M^TD_\pi\theta = \mu(\Sigma_T + \frac\lambda N\Omega)u$, puis
(avec $\Sigma_T = \Sigma_W + \Sigma_B$)

$$\Sigma_Bu = \frac{\mu}{1-\mu}\left(\Sigma_W + \tfrac\lambda N\Omega\right)u.$$

Les $\hat\beta_\ell$ sont donc les directions de la **PDA** : maximiser $u^T\Sigma_Bu$ sous
$u^T(\Sigma_W + \lambda'\Omega)u = 1$ avec $\lambda' = \lambda/N$ (cf. (12.58)). $\square$

**Vérification** sur *waveform* (21 variables ordonnées), avec la pénalité de rugosité.

In [5]:
Xw, gw = datasets.load_waveform("train")
Nw = len(gw)
Om = second_diff_penalty(21)
lam = 50.0
pda = FDA(lam=lam, Omega=Om).fit(Xw, gw)
Yw, _ = class_indicator(gw)
pi_w = Yw.mean(0)
Xc = Xw - Xw.mean(0)
Mw = (Yw.T @ Xc) / Yw.sum(0)[:, None]
SB = Mw.T @ (pi_w[:, None] * Mw)
ST = Xc.T @ Xc / Nw
vals, U = eigh(SB, ST - SB + lam / Nw * Om)
U = U[:, ::-1]
for l in range(2):
    b = pda.beta_[1:, l]
    print("direction %d : |cos(beta_FDA pénalisée, u_PDA)| = %.8f" % (l + 1, abs(b @ U[:, l]) / np.linalg.norm(b) / np.linalg.norm(U[:, l])))
mu_pda = pda.eigvals_
print("mu/(1-mu) =", (mu_pda / (1 - mu_pda)).round(5), "  valeurs propres PDA =", vals[::-1][:2].round(5))

direction 1 : |cos(beta_FDA pénalisée, u_PDA)| = 1.00000000
direction 2 : |cos(beta_FDA pénalisée, u_PDA)| = 1.00000000
mu/(1-mu) = [1.11345 0.74499]   valeurs propres PDA = [1.11345 0.74499]


> **À retenir.** L'*optimal scoring* pénalisé remplace le projecteur $S$ par le lisseur
> $S_\lambda = H(H^TH+\lambda\Omega)^{-1}H^T$ ; ses coefficients sont les directions
> discriminantes de la LDA pénalisée, de métrique $\Sigma_W + \frac\lambda N\Omega$.

## Exercice 12.8 : *Optimal scoring* et directions de la LDA

**Énoncé.** Montrer que les coefficients $\beta_\ell$ obtenus par *optimal scoring* sont
proportionnels aux directions discriminantes $\nu_\ell$ de la LDA.

**Solution.** C'est le cas $\lambda = 0$, $h(x) = x$ de l'exercice précédent. Reprenons
l'argument sans pénalité. Avec les notations de 12.7, $\beta = \Sigma_T^{-1}M^TD_\pi\theta$ et
l'équation aux valeurs propres donne $M\beta = \mu\theta$, d'où

$$\Sigma_B\beta = M^TD_\pi M\beta = \mu M^TD_\pi\theta = \mu\Sigma_T\beta = \mu(\Sigma_W + \Sigma_B)\beta
\quad\Longrightarrow\quad \Sigma_B\beta = \frac{\mu}{1-\mu}\Sigma_W\beta.$$

Or les directions discriminantes de la LDA (section 4.3.3) sont précisément les vecteurs
propres généralisés $\Sigma_B\nu = \gamma\,\Sigma_W\nu$, rangés par $\gamma$ décroissante. Comme
$\mu\mapsto\mu/(1-\mu)$ est croissante sur $[0,1[$, l'ordre est le même ; pour des valeurs
propres distinctes, les espaces propres sont des droites, donc $\beta_\ell\propto\nu_\ell$.
$\square$

Remarque : $\Sigma_W$ normalisée par $N$ ou par $N - K$ ne change que le facteur de
proportionnalité.

In [6]:
fda_v = FDA().fit(Xv, gv)
lda_v = LDA().fit(Xv, gv)
V, gam = lda_v.canonical_directions()
cosines = [abs(fda_v.beta_[1:, l] @ V[:, l]) / np.linalg.norm(fda_v.beta_[1:, l]) / np.linalg.norm(V[:, l])
           for l in range(fda_v.beta_.shape[1])]
print("|cos(beta_l, nu_l)| pour l = 1..%d :" % len(cosines), np.round(cosines, 8))
mu_v = fda_v.eigvals_
print("mu/(1-mu)                   :", (mu_v / (1 - mu_v))[:4].round(4))
print("gamma (W normalisée par N)  :", (gam[:4] * N / (N - K)).round(4))

|cos(beta_l, nu_l)| pour l = 1..10 : [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
mu/(1-mu)                   : [4.052  2.5382 0.3213 0.1381]
gamma (W normalisée par N)  : [4.052  2.5382 0.3213 0.1381]


Les cosinus valent 1 pour les dix directions, et les valeurs propres se correspondent par
$\gamma = \mu/(1-\mu)$ (une fois $W$ normalisée par $N$).

> **À retenir.** Régresser des scores de classes optimaux sur $x$ redonne les directions
> canoniques de Fisher.

## Exercice 12.9 : LDA sur les valeurs ajustées

**Énoncé.** Soit $\hat Y = X\hat B$ la matrice $N\times K$ des valeurs ajustées par la
régression linéaire de la matrice indicatrice sur $X$ ($N\times p$, $p > K$). On considère les
variables réduites $x_i^* = \hat B^Tx_i$. Montrer que la LDA sur les $x_i^*$ est équivalente à
la LDA dans l'espace d'origine.

**Rappel : règle de la LDA.** Avec $\bar\mu$ la moyenne globale et $d_k = \mu_k - \bar\mu$, les
fonctions discriminantes (4.10) s'écrivent, à un terme indépendant de $k$ près,

$$\delta_k(x) = (x - \bar\mu)^TW^{-1}d_k - \tfrac12d_k^TW^{-1}d_k + \log\pi_k .$$

(On développe $x^TW^{-1}\mu_k - \frac12\mu_k^TW^{-1}\mu_k$ avec $\mu_k = \bar\mu + d_k$ ; les
termes $x^TW^{-1}\bar\mu$ et $\bar\mu^TW^{-1}\bar\mu$ ne dépendent pas de $k$.)

**Solution.** *Étape 1 : forme de $\hat B$.* En incluant une constante (ou en centrant), la
partie « pente » de $\hat B$ est $A = \Sigma_T^{-1}M^TD_\pi$ ($p\times K$), avec les notations
de 12.7 : $x^* = c + A^Tx$ où $c$ est un vecteur constant. Les moyennes de classes et la
covariance intra-classe dans l'espace réduit sont donc $\mu_k^* = c + A^T\mu_k$ (donc
$d_k^* = A^Td_k$) et $W^* = A^TWA$.

*Étape 2 : l'image de $A$.* Avec $\Sigma_T = W + M^TD_\pi M$, la formule de
Sherman–Morrison–Woodbury donne
$\Sigma_T^{-1}M^T = W^{-1}M^T(I + D_\pi MW^{-1}M^T)^{-1}$, donc
$A = W^{-1}M^T\,R$ avec $R$ inversible ($K\times K$). Ainsi $\mathrm{Im}(A) = \mathrm{Im}(W^{-1}M^T)$,
qui contient chaque $W^{-1}d_k$.

*Étape 3 : égalité des discriminants.* La matrice $P = A(A^TWA)^+A^TW$ ($^+$ : pseudo-inverse)
est le projecteur $W$-orthogonal sur $\mathrm{Im}(A)$ : il laisse invariant tout vecteur de
$\mathrm{Im}(A)$. Donc $A(A^TWA)^+A^Td_k = P\,W^{-1}d_k = W^{-1}d_k$. Il vient

$$\begin{aligned}
(x^*-\bar\mu^*)^T(W^*)^+d_k^* &= (x-\bar\mu)^TA(A^TWA)^+A^Td_k = (x-\bar\mu)^TW^{-1}d_k,\\
d_k^{*T}(W^*)^+d_k^* &= d_k^TA(A^TWA)^+A^Td_k = d_k^TW^{-1}d_k .
\end{aligned}$$

Les fonctions discriminantes calculées dans l'espace réduit (avec la pseudo-inverse, car
$W^*$ est de rang $K-1$ : les colonnes de $\hat Y$ somment à 1) sont identiques à celles de
l'espace d'origine : mêmes classements. $\square$

In [ ]:
Bhat = np.linalg.lstsq(Hv, Yv, rcond=None)[0]            # (p+1) x K, avec constante
Xstar = Hv @ Bhat                                        # = Y chapeau
lda_orig = LDA().fit(Xv, gv)
lda_red = LDA().fit(Xstar, gv)                           # LDA utilise la pseudo-inverse
Xte, gte = datasets.load_vowel("test")
Xte_star = np.column_stack([np.ones(len(gte)), Xte]) @ Bhat
d_orig = lda_orig.decision_function(Xte)
d_red = lda_red.decision_function(Xte_star)
print("rang de W* :", np.linalg.matrix_rank(lda_red.W_), "(K - 1 =", K - 1, ")")
print("écart des discriminants (à une constante près par point) : %.2e"
      % (d_orig - d_red - (d_orig - d_red).mean(1, keepdims=True)).std(1).max())
print("classements identiques sur le test :", np.array_equal(lda_orig.predict(Xte), lda_red.predict(Xte_star)))
print("erreur test LDA = %.3f" % np.mean(lda_orig.predict(Xte) != gte))

> **À retenir.** La régression sur indicatrices perd de l'information pour *classer
> directement* (masquage, section 4.2), mais ses valeurs ajustées contiennent toute
> l'information utile à la LDA : elles engendrent le sous-espace discriminant.

## Exercice 12.10 : LDA régularisée à noyau

**Énoncé.** On veut faire une LDA à deux classes sur un vecteur de transformations $h(x)$
de grande dimension, avec une covariance intra-classe régularisée $W_h + \gamma I$. Montrer
que le modèle peut être estimé en utilisant uniquement les produits scalaires
$K(x_i,x_{i'}) = \langle h(x_i), h(x_{i'})\rangle$ : la propriété « noyau » des SVM est
partagée par la LDA régularisée.

**Rappels.** Pour deux classes, la direction discriminante est
$\beta = (W_h + \gamma I)^{-1}(m_2 - m_1)$, où $m_k = \frac1{n_k}\sum_{g_i=k}h(x_i)$ et
$W_h = \frac1N\sum_k\sum_{g_i=k}(h(x_i) - m_k)(h(x_i)-m_k)^T$. On classe selon la projection
$h(x)^T\beta$ comparée à un seuil (milieu des moyennes projetées des deux classes).

**Écriture matricielle.** Soit $H$ la matrice des $h(x_i)^T$ et $\mathbf C = I - E$, où $E$
est la matrice de moyennage par classe ($E_{ii'} = 1/n_k$ si $i,i'$ sont dans la classe $k$,
0 sinon). $\mathbf C$ est un projecteur symétrique (centrage par classe), donc
$W_h = \frac1NH^T\mathbf C^T\mathbf CH = \frac1NH^T\mathbf CH$. Enfin $m_2 - m_1 = H^Td$ avec
$d_i = 1/n_2$ si $g_i = 2$, $-1/n_1$ si $g_i = 1$.

**Solution.** *Étape 1 : $\beta$ est dans l'espace engendré par les $h(x_i)$.* De
$(W_h + \gamma I)\beta = H^Td$ on tire $\beta = \frac1\gamma(H^Td - W_h\beta) = H^T\left(\frac1\gamma\left(d - \frac1N\mathbf CH\beta\right)\right)$,
donc $\beta = H^Ta$ pour un $a\in\mathbb R^N$.

*Étape 2 : équation pour $a$.* En remplaçant $\beta = H^Ta$ :
$(W_h + \gamma I)H^Ta = H^T\left(\frac1N\mathbf CHH^Ta + \gamma a\right) = H^T\left(\frac1N\mathbf C\mathbf Ka + \gamma a\right)$,
avec $\mathbf K = HH^T$. Il suffit donc de résoudre le système $N\times N$

$$\left(\tfrac1N\mathbf C\mathbf K + \gamma I\right)a = d .$$

(Toute solution $a$ fournit $\beta = H^Ta$ solution, et $\beta$ est unique car
$W_h + \gamma I$ est inversible.)